# Word Embedding Techniques

This notebook covers four word embedding methods:
1. **CBOW** (from scratch with NumPy)
2. **Skip-Gram** (from scratch with NumPy)
3. **FastText** (using Gensim)
4. **GloVe** (using Gensim pre-trained vectors)

We use an inline text dataset instead of external files so this notebook runs anywhere.

In [ ]:
# Standard libraries only — gensim added for FastText and GloVe
import re
import numpy as np
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print("Libraries ready.")

Libraries ready.


## Step 1: Inline Text Dataset

We use a short text about Artificial Intelligence. In a real project, you would load a large PDF or corpus.

In [ ]:
# Inline text about Artificial Intelligence (replaces external PDF)
raw_text = """
Artificial intelligence is a branch of computer science concerned with building smart machines.
These machines are capable of performing tasks that typically require human intelligence.
Machine learning is a subset of artificial intelligence that enables systems to learn from data.
Deep learning uses neural networks with many layers to learn complex patterns in data.
Natural language processing allows computers to understand and generate human language.
Computer vision enables machines to interpret and understand visual information from images.
Reinforcement learning trains agents to make decisions by rewarding correct actions.
Neural networks are inspired by the structure and function of the human brain.
Training data is essential for teaching machine learning models to make accurate predictions.
Algorithms process large datasets to identify patterns and make intelligent decisions.
Artificial intelligence applications include speech recognition, image classification, and translation.
Supervised learning uses labeled data to train models for classification and regression tasks.
Unsupervised learning discovers hidden patterns in data without using labeled examples.
Transfer learning applies knowledge from one task to improve performance on related tasks.
The transformer architecture revolutionized natural language processing with attention mechanisms.
Large language models are trained on massive text corpora to generate coherent text responses.
"""

print(f"Text loaded: {len(raw_text)} characters")

Text loaded: 1469 characters


## Step 2: Text Preprocessing

Steps: lowercase → remove special characters → tokenize → remove stopwords → lemmatize

In [ ]:
# Step 1: Remove URLs and citation numbers (like [1], [2])
text = re.sub(r'http\S+', '', raw_text)
text = re.sub(r'\[\d+\]', '', text)

# Step 2: Remove everything except alphabets and spaces
text = re.sub(r'[^a-zA-Z\s]', '', text)

# Step 3: Lowercase all text
text = text.lower()

# Step 4: Tokenize into individual words
tokens = word_tokenize(text)

# Step 5: Remove stopwords (common words like 'the', 'is', 'and')
stop_words = set(stopwords.words('english'))
tokens = [word for word in tokens if word not in stop_words]

# Step 6: Lemmatize — reduce words to their base form (e.g., 'learning' → 'learning')
lemmatizer = WordNetLemmatizer()
tokens = [lemmatizer.lemmatize(word) for word in tokens]

# Step 7: Join back into a clean string
clean_text = " ".join(tokens)

print("Cleaned text sample (first 300 chars):")
print(clean_text[:300])
print(f"\nTotal tokens after cleaning: {len(tokens)}")

Cleaned text sample (first 300 chars):
artificial intelligence branch computer science concerned building smart machine machine capable performing task typically require human intelligence machine learning subset artificial intelligence enables system learn data deep learning us neural network many layer learn complex pattern data natura

Total tokens after cleaning: 148


## CBOW — Continuous Bag of Words (from Scratch)

CBOW predicts the **center word** from surrounding context words.  
Architecture: context_words → average embedding → hidden layer → softmax → center word

In [ ]:
# Tokenize the cleaned text into words
words = clean_text.split()

# Build vocabulary: all unique words
vocab = list(dict.fromkeys(words))
vocab_size = len(vocab)

# Word ↔ index mappings for one-hot encoding
word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

print(f"Vocabulary size: {vocab_size}")
print(f"Sample: {vocab[:10]}")

Vocabulary size: 97
Sample: ['artificial', 'intelligence', 'branch', 'computer', 'science', 'concerned', 'building', 'smart', 'machine', 'capable']


In [ ]:
# Generate (context, target) training pairs using sliding window
window_size = 1   # look 1 word to left and right
cbow_data = []

for i in range(window_size, len(words) - window_size):
    context = []
    for j in range(-window_size, window_size + 1):
        if j != 0:
            context.append(word_to_index[words[i + j]])
    target = word_to_index[words[i]]
    cbow_data.append((context, target))

print(f"CBOW training pairs: {len(cbow_data)}")
print("\nSample pairs (context → target):")
for ctx, tgt in cbow_data[:3]:
    ctx_words = [index_to_word[i] for i in ctx]
    tgt_word = index_to_word[tgt]
    print(f"  {ctx_words} → '{tgt_word}'")

CBOW training pairs: 146

Sample pairs (context → target):
  ['artificial', 'branch'] → 'intelligence'
  ['intelligence', 'computer'] → 'branch'
  ['branch', 'science'] → 'computer'


In [ ]:
np.random.seed(42)
embedding_dim = 5   # word vector size (small for demonstration)

# W1: embedding matrix — row i = vector for word i
W1_cbow = np.random.randn(vocab_size, embedding_dim) * 0.01

# W2: projection matrix — maps embedding to vocabulary scores
W2_cbow = np.random.randn(embedding_dim, vocab_size) * 0.01

learning_rate = 0.01

def softmax(x):
    # Numerically stable softmax: subtract max before exponentiation
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum()

print(f"W1 shape: {W1_cbow.shape} | W2 shape: {W2_cbow.shape}")

W1 shape: (97, 5) | W2 shape: (5, 97)


In [ ]:
# CBOW training loop
epochs = 1000
for epoch in range(epochs):
    loss = 0
    for context_indices, target_index in cbow_data:

        # Forward: average context word embeddings
        h = np.mean(W1_cbow[context_indices], axis=0)

        # Project to vocabulary and apply softmax
        u = np.dot(h, W2_cbow)
        y_pred = softmax(u)

        # Cross-entropy loss
        loss -= np.log(y_pred[target_index] + 1e-9)

        # Backprop: compute gradients
        error = y_pred.copy()
        error[target_index] -= 1
        dW2 = np.outer(h, error)
        dW1 = np.dot(W2_cbow, error) / len(context_indices)

        # Update weights
        W2_cbow -= learning_rate * dW2
        for idx in context_indices:
            W1_cbow[idx] -= learning_rate * dW1

if epoch % 200 == 0:
    print(f"Epoch {epoch}, Loss: {loss:.4f}")

print("\nCBOW training done.")


CBOW training done.


In [ ]:
print("CBOW Word Embeddings (W1 rows):\n")
for word in vocab[:8]:
    vec = W1_cbow[word_to_index[word]]
    print(f"{word} → {vec}")

CBOW Word Embeddings (W1 rows):

artificial → [ 3.48177907  3.91332875  1.55128681  0.59847898 -1.64652488]
intelligence → [-3.88843182  0.64694614  5.14000769 -1.92272066 -1.55546674]
branch → [ 1.6412833   1.41956891  1.37293777 -0.6412534  -1.86591282]
computer → [-1.01780827  3.59586947  0.44782849 -4.71328386  2.11583696]
science → [ 2.20015093 -0.72642437 -0.13794422 -2.88960354 -0.77090202]
concerned → [ 0.42747849 -0.31742633 -0.36379158 -1.5593576  -0.14079182]
building → [ 1.0565678  -0.1976455  -0.03440807 -0.3475233   0.64707144]
smart → [-0.13676623  0.43955841  0.44596866 -1.03353324 -3.49611423]


## Skip-Gram (from Scratch)

Skip-Gram predicts **context words** from the center word.  
Each center word creates 2×window_size training pairs.

In [ ]:
# Generate (center, context) pairs — one pair per (center, neighbor) combination
skipgram_data = []

for i in range(window_size, len(words) - window_size):
    center = word_to_index[words[i]]
    for j in range(-window_size, window_size + 1):
        if j != 0:
            context = word_to_index[words[i + j]]
            skipgram_data.append((center, context))

print(f"Skip-Gram training pairs: {len(skipgram_data)}")
print(f"(More pairs than CBOW because each center creates {window_size*2} separate samples)")

Skip-Gram training pairs: 292
(More pairs than CBOW because each center creates 2 separate samples)


In [ ]:
np.random.seed(42)

# Fresh weight matrices for Skip-Gram
W1_sg = np.random.randn(vocab_size, embedding_dim) * 0.01
W2_sg = np.random.randn(embedding_dim, vocab_size) * 0.01

# Training loop for Skip-Gram
for epoch in range(epochs):
    loss = 0
    for center_index, context_index in skipgram_data:

        # Forward: look up center word embedding
        h = W1_sg[center_index]

        # Project to vocabulary and apply softmax
        u = np.dot(h, W2_sg)
        y_pred = softmax(u)

        # Cross-entropy loss for this context word
        loss -= np.log(y_pred[context_index] + 1e-9)

        # Backprop
        error = y_pred.copy()
        error[context_index] -= 1
        dW2 = np.outer(h, error)
        dW1 = np.dot(W2_sg, error)

        # Update only the center word's row in W1
        W2_sg -= learning_rate * dW2
        W1_sg[center_index] -= learning_rate * dW1

if epoch % 200 == 0:
    print(f"Epoch {epoch}, Loss: {loss:.4f}")

print("\nSkip-Gram training done.")


Skip-Gram training done.


In [ ]:
print("Skip-Gram Word Embeddings (W1 rows):\n")
for word in vocab[:8]:
    vec = W1_sg[word_to_index[word]]
    print(f"{word} → {vec}")

Skip-Gram Word Embeddings (W1 rows):

artificial → [ 2.65812224  3.37177511 -2.05696387 -0.37626893 -1.67045319]
intelligence → [-3.60781136  1.17120791  2.76280403 -1.85789209 -0.9587761 ]
branch → [ 2.92488428  1.42656508  1.79955856 -1.20531491 -2.81128862]
computer → [ 0.17543802  4.38701405  0.95037208 -2.64426667  0.39284863]
science → [ 2.56163386  0.72015818  2.39981013 -1.82235014 -0.88644373]
concerned → [ 0.07755623  2.64883921  2.55724321 -1.88059069 -0.8444406 ]
building → [ 2.21897739  1.97403002  2.99863394 -0.97878887 -0.14487944]
smart → [-0.0986104   1.60081496  2.03928145 -1.27665431 -1.77989216]


## FastText (using Gensim)

FastText extends Word2Vec by also modeling **character-level subwords**. This lets it produce vectors for words not seen during training (OOV words).

In [ ]:
!pip install gensim -q
from gensim.models import FastText

# Prepare sentence-level input: list of token lists
ft_sentences = [sent.split() for sent in raw_text.strip().split('\n') if len(sent.strip()) > 5]

# Train FastText model
ft_model = FastText(
    sentences=ft_sentences,
    vector_size=50,   # 50-dimensional word vectors
    window=3,         # context window: 3 words on each side
    min_count=1,      # include all words regardless of frequency
    epochs=100        # training passes over the data
)

print("FastText model trained.")
print(f"Vocabulary size: {len(ft_model.wv.key_to_index)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.0 MB/s eta 0:00:00
FastText model trained.
Vocabulary size: 131


In [ ]:
# Get vector for a known word
vector = ft_model.wv["learning"]
print(f"Vector for 'learning' (first 10 dims): {vector[:10]}")
print(f"Vector shape: {vector.shape}")

Vector for 'learning' (first 10 dims): [-0.02378331 -0.01495385 -0.03161469  0.02328509 -0.01733052  0.01454969
  0.03921294  0.02676196 -0.07828765  0.02765628]
Vector shape: (50,)


In [ ]:
# FastText OOV example — 'learner' was not in training data
# but FastText builds it from character n-grams of 'learn', 'earn', 'rner', etc.
oov_vector = ft_model.wv["learner"]
print(f"OOV vector for 'learner' (first 10 dims): {oov_vector[:10]}")
print("(This works because FastText uses character n-grams, unlike Word2Vec which would fail here)")

OOV vector for 'learner' (first 10 dims): [-0.01159297 -0.0087575  -0.01595333  0.01325792 -0.00469468  0.00940013
  0.01969416  0.01331708 -0.04530583  0.01524756]
(This works because FastText uses character n-grams, unlike Word2Vec which would fail here)


In [ ]:
# Word similarity using FastText
similarity = ft_model.wv.similarity("machine", "deep")
print(f"Similarity between 'machine' and 'deep': {similarity:.4f}")

similarity2 = ft_model.wv.similarity("learning", "training")
print(f"Similarity between 'learning' and 'training': {similarity2:.4f}")

Similarity between 'machine' and 'deep': 0.5863
Similarity between 'learning' and 'training': 0.9922


## GloVe (using Gensim Pre-trained Vectors)

GloVe (Global Vectors) is trained on global word co-occurrence statistics from large corpora. We use Gensim's downloader for the pre-trained model, with a demo fallback for offline environments.

In [ ]:
import gensim.downloader as api
from gensim.models import KeyedVectors

def build_demo_glove():
    # Compact demo vectors with hand-crafted semantic structure
    dim = 50
    def vec(royalty, gender, animal, geo, sentiment, seed):
        np.random.seed(seed)
        v = np.random.normal(0, 0.05, dim).astype(np.float32)
        v[0], v[1], v[2], v[3], v[4] = royalty, gender, animal, geo, sentiment
        return v
    words = {
        "machine":  vec( 0.0,  0.0, -0.1, 0.0,  0.3,  1),
        "learning": vec( 0.0,  0.0, -0.1, 0.0,  0.3,  2),
        "deep":     vec( 0.1,  0.0, -0.1, 0.0,  0.4,  3),
        "neural":   vec( 0.1,  0.0, -0.1, 0.0,  0.3,  4),
        "data":     vec(-0.1,  0.0,  0.0, 0.0,  0.2,  5),
        "king":     vec( 0.9,  0.8, -0.1, 0.0,  0.0,  6),
        "queen":    vec( 0.9, -0.8, -0.1, 0.0,  0.0,  7),
        "man":      vec( 0.0,  0.9,  0.0, 0.0,  0.0,  8),
        "woman":    vec( 0.0, -0.9,  0.0, 0.0,  0.0,  9),
    }
    kv = KeyedVectors(vector_size=dim)
    kv.add_vectors(list(words.keys()), list(words.values()))
    return kv

try:
    # Try loading the real 50-dim GloVe trained on Wikipedia + Gigaword
    glove = api.load("glove-wiki-gigaword-50")
    print(f"Real GloVe loaded | vocab: {len(glove):,} | dims: {glove.vector_size}")
except Exception:
    glove = build_demo_glove()
    print(f"Demo GloVe loaded | vocab: {len(glove)} | dims: {glove.vector_size}")

[==================================================] 100.0% 66.0/66.0MB downloaded
Real GloVe loaded | vocab: 400,000 | dims: 50


In [ ]:
# Get vector for the word 'learning'
learning_vec = glove['learning']
print(f"Vector for 'learning': {learning_vec[:10]} ...")
print(f"Vector shape: {learning_vec.shape}")

Vector for 'learning': [ 0.20461   0.48659  -0.55308  -0.27019   0.26336   0.15751  -0.28994
 -0.51824   0.051829  0.36225 ] ...
Vector shape: (50,)


In [ ]:
# Cosine similarity between 'machine' and 'deep'
sim = glove.similarity('machine', 'deep')
print(f"GloVe similarity between 'machine' and 'deep': {sim:.4f}")

# More similarity examples
for w1, w2 in [('machine', 'learning'), ('neural', 'deep'), ('data', 'machine')]:
    try:
        s = glove.similarity(w1, w2)
        print(f"similarity('{w1}', '{w2}') = {s:.4f}")
    except KeyError as e:
        print(f"Word not in vocab: {e}")

GloVe similarity between 'machine' and 'deep': 0.3715
similarity('machine', 'learning') = 0.4803
similarity('neural', 'deep') = 0.3183
similarity('data', 'machine') = 0.5019


## Comparison Summary

| Method | Library | OOV Handling | Training |
|--------|---------|--------------|----------|
| CBOW | NumPy (from scratch) | No | Predict center from context |
| Skip-Gram | NumPy (from scratch) | No | Predict context from center |
| FastText | Gensim | Yes (char n-grams) | Word + subword |
| GloVe | Gensim (pre-trained) | No | Global co-occurrence matrix |

- **CBOW/Skip-Gram from scratch**: best for understanding how embeddings are actually learned
- **FastText**: best when OOV words are common (e.g., names, technical terms, new words)
- **GloVe**: excellent pre-trained vectors, especially for semantic similarity and analogy tasks